# 01 — Bronze exploration

Load official January 2025 TLC Yellow Taxi Parquet and taxi-zone reference CSV. Audit schema, missingness, date coverage and anomalies. On Fabric, attach `LH_NYC_Taxi` as the default Lakehouse.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

## Load data

If using local Spark instead of Fabric, change `Files/...` to `data/raw/...` and `data/reference/...`. The separate `scripts/download_data.py` creates the local files.

In [ ]:
from pyspark.sql import functions as F
from pathlib import Path

BRONZE_PATH = "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
LOOKUP_PATH = "Files/reference/taxi_zone_lookup.csv"
# Local Spark alternative:
# BRONZE_PATH = "data/raw/yellow_tripdata_2025-01.parquet"
# LOOKUP_PATH = "data/reference/taxi_zone_lookup.csv"
df = spark.read.parquet(BRONZE_PATH)
zone_df = spark.read.option("header",True).option("inferSchema",True).csv(LOOKUP_PATH)
print("Trips:", df.count(), "columns:", len(df.columns), "zone rows:", zone_df.count())
df.printSchema()

In [ ]:
df.select(F.min("tpep_pickup_datetime").alias("first_pickup"),F.max("tpep_pickup_datetime").alias("last_pickup")).show(truncate=False)
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

In [ ]:
df.select(F.min("trip_distance").alias("min_miles"),F.max("trip_distance").alias("max_miles"),F.expr("percentile_approx(trip_distance, 0.5)").alias("median_miles")).show()
df.withColumn("trip_minutes", F.expr("timestampdiff(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime)/60.0")) \
 .select(F.sum((F.col("trip_minutes") < 0).cast("int")).alias("negative_durations"),F.sum((F.col("trip_minutes") == 0).cast("int")).alias("zero_durations")).show()

In [ ]:
print("Duplicate complete records:",df.count()-df.dropDuplicates().count())
print("Unmatched pickup zone IDs:",df.join(zone_df,df.PULocationID==zone_df.LocationID,"left_anti").count())
zone_df.filter(F.col("LocationID").isin(264,265)).show(truncate=False)

**Known observations from previous Fabric execution:** 3,475,226 records; 21 pickups before January and one after; no unmatched pickup zones and no exact duplicate records. Anomalies should be flagged rather than automatically discarded. See `docs/methodology.md` and the original notebook.